# Ensemble Methods on AirBnB 🦾🦾

# Faire du voting et du stacking !

Two popular boosting algorithms are Adaboost and XGBoost, the goal of this exercise is to apply them both to a prediction problem and evaluate their performance with different base models. The dataset we will use is that of Airbnb listings in Seattle, the goal is to predict the price per night of the listing.
There will be quite a lot of preprocessing to do in this exercise as well as some interesting exploratory analysis and visualization. Do not hesitate to deviate from the questions to explore the data further.

1. Let's import the usual librairies.

In [2]:
#!pip install -q xgboost
#!pip install -q s3fs
 #!pip install -U kaleido

# Load in our libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "colab" # pour que colab ne bloque pas l'export svg

import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer, KNNImputer

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import r2_score, accuracy_score

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# import ensemble methods
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, AdaBoostRegressor, GradientBoostingClassifier, VotingRegressor, StackingClassifier,  StackingRegressor

from sklearn.svm import SVR

from xgboost import XGBRegressor, XGBClassifier





2. Import the ```listings.csv``` dataset from s3 using the following link: https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/Boosting/listings.csv

In [3]:
url = "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/Boosting/listings.csv"

df = pd.read_csv(url)

df.head()

,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,review_scores_value,requires_license,license,jurisdiction_names,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month
0,241032,https://www.airbnb.com/rooms/241032,20160104002432,2016-01-04,Stylish Queen Anne Apartment,NaN,Make your self at home in this charming one-be...,Make your self at home in this charming one-be...,none,NaN,...,10.0,f,NaN,WASHINGTON,f,moderate,f,f,2,4.07
1,953595,https://www.airbnb.com/rooms/953595,20160104002432,2016-01-04,Bright & Airy Queen Anne Apartment,Chemically sensitive? We've removed the irrita...,"Beautiful, hypoallergenic apartment in an extr...",Chemically sensitive? We've removed the irrita...,none,"Queen Anne is a wonderful, truly functional vi...",...,10.0,f,NaN,WASHINGTON,f,strict,t,t,6,1.48
2,3308979,https://www.airbnb.com/rooms/3308979,20160104002432,2016-01-04,New Modern House-Amazing water view,New modern house built in 2013. Spectacular s...,"Our house is modern, light and fresh with a wa...",New modern house built in 2013. Spectacular s...,none,Upper Queen Anne is a charming neighborhood fu...,...,10.0,f,NaN,WASHINGTON,f,strict,f,f,2,1.15
3,7421966,https://www.airbnb.com/rooms/7421966,20160104002432,2016-01-04,Queen Anne Chateau,A charming apartment that sits atop Queen Anne...,NaN,A charming apartment that sits atop Queen Anne...,none,NaN,...,NaN,f,NaN,WASHINGTON,f,flexible,f,f,1,NaN
4,278830,https://www.airbnb.com/rooms/278830,20160104002432,2016-01-04,Charming craftsman 3 bdm house,Cozy family craftman house in beautiful neighb...,Cozy family craftman house in beautiful neighb...,Cozy family craftman house in beautiful neighb...,none,We are in the beautiful neighborhood of Queen ...,...,9.0,f,NaN,WASHINGTON,f,strict,f,f,1,0.89


3. There are a lot of columns in this dataset. Display the dataset info.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 92 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id                                3818 non-null   int64  
 1   listing_url                       3818 non-null   object 
 2   scrape_id                         3818 non-null   int64  
 3   last_scraped                      3818 non-null   object 
 4   name                              3818 non-null   object 
 5   summary                           3641 non-null   object 
 6   space                             3249 non-null   object 
 7   description                       3818 non-null   object 
 8   experiences_offered               3818 non-null   object 
 9   neighborhood_overview             2786 non-null   object 
 10  notes                             2212 non-null   object 
 11  transit                           2884 non-null   object 
 12  thumbn

4. Let's proceed to some visualization, first display the distribution of the price variable. You will have to preprocess it as it is not in a numerical format.

In [5]:
df["price"]

,price
0,$85.00
1,$150.00
2,$975.00
3,$100.00
4,$450.00
...,...
3813,$359.00
3814,$79.00
3815,$93.00
3816,$99.00


In [6]:
df["price"].describe()

,price
count,3818
unique,273
top,$150.00
freq,162


In [7]:
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    # $1,000 sinon astype(float) could not convert string to float: '1,000.00'
    .astype(float)
)

df["price"].head()


,price
0,85.0
1,150.0
2,975.0
3,100.0
4,450.0


In [8]:


fig = px.histogram(
    df,
    x="price",
    nbins=50,
    title="Distribution of Airbnb Prices",
)

fig.update_layout(
    xaxis_title="Price ($)",
    yaxis_title="Frequency",
    bargap=0.05
)

fig.show()

5. The distribution of the target variable is skewed towards high values (this is a very usual situation when working with prices, many items are around the average price range and the higher the price, the fewer items there are). A standard way of working with such variables is to change the scale using the log function so the distribution becomes evenly distributed.
Create a price_log variable that's equal to log(price)

In [9]:
df["price"].describe()

,price
count,3818.000000
mean,127.976166
std,90.250022
min,20.000000
25%,75.000000
50%,100.000000
75%,150.000000
max,1000.000000


In [10]:
df["price_log"] = np.log10(df["price"]) # si min = 0 => .replace(0, np.nan) sinon log10 échouera, mais ce n'est pas le cas ici
df["price_log"].describe()


,price_log
count,3818.000000
mean,2.032163
std,0.245605
min,1.301030
25%,1.875061
50%,2.000000
75%,2.176091
max,3.000000


In [11]:

pio.renderers.default = "colab" # pour que colab ne bloque pas l'export svg
fig = px.histogram(
    df,
    x="price_log",
    nbins=50,
    title="Distribution of Airbnb Log Prices",
)

fig.update_layout(
    xaxis_title="Log Price ($)",
    yaxis_title="Frequency",
    bargap=0.05
)

fig.show()

The distribution looks a lot better for prediction purposes after the log transformation!

6. Visualize the price against the following variables :

- ```room type```
- ```beds```
- ```property type```

In [12]:
df[["room_type", "beds", "property_type"]].describe(include="all")
# df[["","",""]]!!! /  sans include="all" les stats sur toutes les var (cat et num) ne s'afficheront pas

,room_type,beds,property_type
count,3818,3817.000000,3817
unique,3,NaN,16
top,Entire home/apt,NaN,House
freq,2541,NaN,1733
mean,NaN,1.735394,NaN
std,NaN,1.139480,NaN
min,NaN,1.000000,NaN
25%,NaN,1.000000,NaN
50%,NaN,1.000000,NaN
75%,NaN,2.000000,NaN


In [13]:
fig = px.box(
    df,
    x="room_type",
    y="price_log",
    title="Price Distribution by Room Type",
    points="all"
)
fig.show()


In [14]:
fig = px.box(
    df,
    x="beds",
    y="price_log",
    title="Price Distribution by Number of Beds"
)
fig.show()


In [15]:
fig = px.scatter(
    df,
    x="beds",
    y="price_log",
    title="Price vs Number of Beds",
    trendline="lowess",
    opacity=0.5
)
fig.show()


In [16]:
fig = px.box(
    df,
    x="property_type",
    y="price_log",
    title="Price Distribution by Property Type"
)
fig.update_layout(xaxis={'categoryorder':'median descending'})
fig.show()


7. Isolate the target variable in an object y and the other variables in an object X

In [17]:
# Target variable
y = df["price_log"]

# Features (all columns except price_log)
X = df.drop(columns=["price_log"])


8. We will have to remove a certain number of variables that we do not know how to use at this point. Start by removing the variables that could be interpreted as an ```id``` , we will also remove the variables that contain long texts as we haven't learned about text processing yet.

We also have to remove all variables related to price, as they represent a risk of leak because of their direct link to the target variable, like ```monthly price```.

A certain number of variables contain a very high amount of missing values, in some cases these missing values correspond to an information we can exploit, sometimes not. Remove these not so useful variables from the dataset, strat by checking the proportion of missing values for all variables.

Your dataset should only contain categorical and numerical variables after this step. Check if your final dataset contains the following variables :

```
Index(['host_response_time', 'host_response_rate',
       'host_acceptance_rate', 'host_is_superhost', 'host_listings_count',
       'host_total_listings_count', 'host_has_profile_pic',
       'host_identity_verified', 'neighbourhood_group_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bedrooms', 'beds', 'bed_type', 'security_deposit', 'cleaning_fee',
       'guests_included', 'extra_people', 'minimum_nights', 'maximum_nights',
       'has_availability', 'availability_30', 'availability_60',
       'availability_90', 'availability_365', 'number_of_reviews',
       'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value', 'requires_license', 'instant_bookable',
       'cancellation_policy', 'require_guest_profile_picture',
       'require_guest_phone_verification', 'calculated_host_listings_count',
       'reviews_per_month'],
      dtype='object')
```

In [18]:
selected_features = [
    'host_response_time', 'host_response_rate',
    'host_acceptance_rate', 'host_is_superhost', 'host_listings_count',
    'host_total_listings_count', 'host_has_profile_pic',
    'host_identity_verified', 'neighbourhood_group_cleansed', 'latitude',
    'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
    'bedrooms', 'beds', 'bed_type', 'security_deposit', 'cleaning_fee',
    'guests_included', 'extra_people', 'minimum_nights', 'maximum_nights',
    'has_availability', 'availability_30', 'availability_60',
    'availability_90', 'availability_365', 'number_of_reviews',
    'review_scores_rating', 'review_scores_accuracy',
    'review_scores_cleanliness', 'review_scores_checkin',
    'review_scores_communication', 'review_scores_location',
    'review_scores_value', 'requires_license', 'instant_bookable',
    'cancellation_policy', 'require_guest_profile_picture',
    'require_guest_phone_verification', 'calculated_host_listings_count',
    'reviews_per_month'
]




In [19]:
selected_columns = selected_features + ["price_log"]
df_clean = df[selected_columns].copy()



9. Are there any remaining missing values ? Is there a relevant way to replace those missing values without using imputing methods ? Are all the variables in a numerical format ? If not run some preprocessing to create a clean dataset.

In [20]:
df_clean.describe(include="all")

,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood_group_cleansed,latitude,...,review_scores_location,review_scores_value,requires_license,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month,price_log
count,3295,3295,3045,3816,3816.000000,3816.000000,3816,3816,3818,3818.000000,...,3163.000000,3162.000000,3818,3818,3818,3818,3818,3818.000000,3191.000000,3818.000000
unique,4,45,2,2,NaN,NaN,2,2,17,NaN,...,NaN,NaN,1,2,3,2,2,NaN,NaN,NaN
top,within an hour,100%,100%,f,NaN,NaN,t,t,Other neighborhoods,NaN,...,NaN,NaN,f,f,strict,f,f,NaN,NaN,NaN
freq,1692,2371,3044,3038,NaN,NaN,3809,2997,794,NaN,...,NaN,NaN,3818,3227,1417,3497,3443,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,7.157757,7.157757,NaN,NaN,NaN,47.628961,...,9.608916,9.452245,NaN,NaN,NaN,NaN,NaN,2.946307,2.078919,2.032163
std,NaN,NaN,NaN,NaN,28.628149,28.628149,NaN,NaN,NaN,0.043052,...,0.629053,0.750259,NaN,NaN,NaN,NaN,NaN,5.893029,1.822348,0.245605
min,NaN,NaN,NaN,NaN,1.000000,1.000000,NaN,NaN,NaN,47.505088,...,4.000000,2.000000,NaN,NaN,NaN,NaN,NaN,1.000000,0.020000,1.301030
25%,NaN,NaN,NaN,NaN,1.000000,1.000000,NaN,NaN,NaN,47.609418,...,9.000000,9.000000,NaN,NaN,NaN,NaN,NaN,1.000000,0.695000,1.875061
50%,NaN,NaN,NaN,NaN,1.000000,1.000000,NaN,NaN,NaN,47.623601,...,10.000000,10.000000,NaN,NaN,NaN,NaN,NaN,1.000000,1.540000,2.000000
75%,NaN,NaN,NaN,NaN,3.000000,3.000000,NaN,NaN,NaN,47.662694,...,10.000000,10.000000,NaN,NaN,NaN,NaN,NaN,2.000000,3.000000,2.176091


In [21]:
na_percent = df_clean.isna().mean() * 100
na_percent[na_percent > 0].sort_index() # .sort_values(ascending=False)



,0
bathrooms,0.419068
bedrooms,0.157150
beds,0.026192
cleaning_fee,26.977475
host_acceptance_rate,20.246202
host_has_profile_pic,0.052383
host_identity_verified,0.052383
host_is_superhost,0.052383
host_listings_count,0.052383
host_response_rate,13.698271


In [22]:
(na_percent>0).sum()

np.int64(22)

In [23]:
missing_cols = set(na_percent[na_percent > 0].index)
missing_cols

{'bathrooms',
 'bedrooms',
 'beds',
 'cleaning_fee',
 'host_acceptance_rate',
 'host_has_profile_pic',
 'host_identity_verified',
 'host_is_superhost',
 'host_listings_count',
 'host_response_rate',
 'host_response_time',
 'host_total_listings_count',
 'property_type',
 'review_scores_accuracy',
 'review_scores_checkin',
 'review_scores_cleanliness',
 'review_scores_communication',
 'review_scores_location',
 'review_scores_rating',
 'review_scores_value',
 'reviews_per_month',
 'security_deposit'}

In [24]:
# occupations : not specified => 0
for col in ['bathrooms', 'bedrooms', 'beds']:
    df_clean[col] = df_clean[col].fillna(0)


In [25]:
treated = {'bathrooms', 'bedrooms', 'beds'}
missing_cols = missing_cols - treated
missing_cols

{'cleaning_fee',
 'host_acceptance_rate',
 'host_has_profile_pic',
 'host_identity_verified',
 'host_is_superhost',
 'host_listings_count',
 'host_response_rate',
 'host_response_time',
 'host_total_listings_count',
 'property_type',
 'review_scores_accuracy',
 'review_scores_checkin',
 'review_scores_cleanliness',
 'review_scores_communication',
 'review_scores_location',
 'review_scores_rating',
 'review_scores_value',
 'reviews_per_month',
 'security_deposit'}

In [26]:
# binary => yes no
binary_cols = [
    'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',


]

for col in binary_cols:
    df_clean[col] = df_clean[col].fillna("f")



In [27]:
treated = {'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified'}
missing_cols = missing_cols - treated
missing_cols

{'cleaning_fee',
 'host_acceptance_rate',
 'host_listings_count',
 'host_response_rate',
 'host_response_time',
 'host_total_listings_count',
 'property_type',
 'review_scores_accuracy',
 'review_scores_checkin',
 'review_scores_cleanliness',
 'review_scores_communication',
 'review_scores_location',
 'review_scores_rating',
 'review_scores_value',
 'reviews_per_month',
 'security_deposit'}

In [28]:
# no review => 0
review_cols = [
    'review_scores_rating', 'review_scores_accuracy',
    'review_scores_cleanliness', 'review_scores_checkin',
    'review_scores_communication', 'review_scores_location',
    'review_scores_value', 'reviews_per_month'
]

for col in review_cols:
    df_clean[col] = df_clean[col].fillna(0)


In [29]:
treated = {'review_scores_rating', 'review_scores_accuracy',
    'review_scores_cleanliness', 'review_scores_checkin',
    'review_scores_communication', 'review_scores_location',
    'review_scores_value', 'reviews_per_month'}
missing_cols = missing_cols - treated
missing_cols

{'cleaning_fee',
 'host_acceptance_rate',
 'host_listings_count',
 'host_response_rate',
 'host_response_time',
 'host_total_listings_count',
 'property_type',
 'security_deposit'}

In [30]:
# missing monetary values => no fee => so 0
money_cols = ['security_deposit', 'cleaning_fee']
for col in money_cols:
    df_clean[col] = df_clean[col].fillna(0)


In [31]:
treated = {'security_deposit', 'cleaning_fee'}
missing_cols = missing_cols - treated
missing_cols


{'host_acceptance_rate',
 'host_listings_count',
 'host_response_rate',
 'host_response_time',
 'host_total_listings_count',
 'property_type'}

In [32]:
df_clean['host_response_time'] = df_clean['host_response_time'].fillna(0)
df_clean['host_response_rate'] = df_clean['host_response_rate'].fillna(0)
df_clean['host_acceptance_rate'] = df_clean['host_acceptance_rate'].fillna(0)


In [33]:
treated = {'host_response_time', 'host_response_rate', 'host_acceptance_rate'}
missing_cols = missing_cols - treated
missing_cols

{'host_listings_count', 'host_total_listings_count', 'property_type'}

In [34]:
df_clean['host_listings_count'] = df_clean['host_listings_count'].fillna(1)
df_clean['host_total_listings_count'] = df_clean['host_total_listings_count'].fillna(1)
# hôte sans information sur le nombre de listings = hôte avec 1 seul logemnt
# mode = 1 dans les stats descr
df_clean['property_type'] = df_clean['property_type'].fillna("Apartment")
# ne pas créer de catégorie artificielle avec unknown par
# par défaut property non renseigné => Apartment

In [35]:
treated = {'host_listings_count', 'host_total_listings_count', 'property_type'}
missing_cols = missing_cols - treated
missing_cols

set()

In [36]:
df_clean.isna().sum().sum()

np.int64(0)

# check numerical variables

In [59]:
columns_to_convert =pd.DataFrame((df_clean.dtypes[df_clean.dtypes == 'object']))
columns_to_convert.describe()

,0
count,19
unique,1
top,object
freq,19


In [61]:
# objectif pédagogique : montrer comment ttansformer un dataset en 100% numérique sans pipeline
# df_final # df_clean => utiliser df_clean dans le pipeline non pas df_final sinon pipeline sans intérêt
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
df_final =  pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)
# get.dummies () => one hot encoding
df_final.describe()

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,guests_included,minimum_nights,...,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,reviews_per_month,price_log
count,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,...,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000,3818.000000
mean,7.154531,7.154531,47.628961,-122.333103,3.349398,1.254191,1.305657,1.734940,1.672603,2.369303,...,78.518596,7.975642,7.921949,8.100052,8.136983,7.960450,7.828182,2.946307,1.737514,2.032163
std,28.620995,28.620995,0.043052,0.031745,1.977599,0.594724,0.884219,1.139677,1.311040,16.305902,...,35.979061,3.694899,3.671283,3.736184,3.725777,3.667938,3.630832,5.893029,1.835425,0.245605
min,1.000000,1.000000,47.505088,-122.417219,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.301030
25%,1.000000,1.000000,47.609418,-122.354320,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,87.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,1.000000,0.310000,1.875061
50%,1.000000,1.000000,47.623601,-122.328874,3.000000,1.000000,1.000000,1.000000,1.000000,2.000000,...,95.000000,10.000000,10.000000,10.000000,10.000000,10.000000,9.000000,1.000000,1.105000,2.000000
75%,3.000000,3.000000,47.662694,-122.310800,4.000000,1.000000,2.000000,2.000000,2.000000,2.000000,...,98.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,2.000000,2.660000,2.176091
max,502.000000,502.000000,47.733358,-122.240607,16.000000,8.000000,7.000000,15.000000,15.000000,1000.000000,...,100.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,37.000000,12.150000,3.000000


10. Check that all variables that can can be converted are in numerical format, do not forget to check y as well.

In [64]:
non_numeric_cols = df_final.dtypes[df_final.dtypes == "object"]
non_numeric_cols

,0


In [65]:
y = pd.to_numeric(y, errors="coerce")
print(y.dtype)



float64


11. Apply ```train_test_split``` to create an X_train X_test y_train and y_test objects. (with random_state = 1)

In [68]:
X = df_clean.drop(columns=["price_log"])
y = df_clean["price_log"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1
)


12. Separate the variables into two groups, one for the numerical variables and one for the categorical variables. And apply preprocessings to each subgroup of variables properly.

In [69]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_features


['host_listings_count',
 'host_total_listings_count',
 'latitude',
 'longitude',
 'accommodates',
 'bathrooms',
 'bedrooms',
 'beds',
 'guests_included',
 'minimum_nights',
 'maximum_nights',
 'availability_30',
 'availability_60',
 'availability_90',
 'availability_365',
 'number_of_reviews',
 'review_scores_rating',
 'review_scores_accuracy',
 'review_scores_cleanliness',
 'review_scores_checkin',
 'review_scores_communication',
 'review_scores_location',
 'review_scores_value',
 'calculated_host_listings_count',
 'reviews_per_month']

In [70]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
categorical_features


['host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood_group_cleansed',
 'property_type',
 'room_type',
 'bed_type',
 'security_deposit',
 'cleaning_fee',
 'extra_people',
 'has_availability',
 'requires_license',
 'instant_bookable',
 'cancellation_policy',
 'require_guest_profile_picture',
 'require_guest_phone_verification']

In [71]:
# onehot encoder de scikit-learn exige que chaque var catégorielle soit exactement de type string
df_clean[categorical_features] = df_clean[categorical_features].astype(str)

In [72]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 45 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   host_response_time                3818 non-null   object 
 1   host_response_rate                3818 non-null   object 
 2   host_acceptance_rate              3818 non-null   object 
 3   host_is_superhost                 3818 non-null   object 
 4   host_listings_count               3818 non-null   float64
 5   host_total_listings_count         3818 non-null   float64
 6   host_has_profile_pic              3818 non-null   object 
 7   host_identity_verified            3818 non-null   object 
 8   neighbourhood_group_cleansed      3818 non-null   object 
 9   latitude                          3818 non-null   float64
 10  longitude                         3818 non-null   float64
 11  property_type                     3818 non-null   object 
 12  room_t

In [73]:


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


13. What score would you expect for a model that would always predict the average price?

In [74]:


# Baseline prediction: always predict the mean of y_train
y_pred_train = np.full_like(y_train, y_train.mean(), dtype=float)
y_pred_test = np.full_like(y_test, y_train.mean(), dtype=float)

# Compute R² scores
r2_train = r2_score(y_train, y_pred_train)
# or y_pred_train_i = y_train.mean_i => r² = 1- (somme(y_train_i - y_pred_train_i)²/somme(y_train_i - y_train.mean_i)²) => 1-1=0
# Rappel :  Somme des Carrés des Résidus SCR = somme(y_train_i - y_pred_train_i)²
# Somme Totale des Carrés STC = somme(y_train_i - y_train.mean_i)²
r2_test = r2_score(y_test, y_pred_test)
# or le modèle prédit la moyenne du train pas celle du test
# sachant que les distrib train et test ne sont pas les m^mes
# SCR > STC => R² = 1 - (SCR/STC) => R²<0
print("R2 baseline on training set :", r2_train)
print("R2 baseline on test set :", r2_test)




R2 baseline on training set : 0.0
R2 baseline on test set : -0.0001401602016899428


14. Train an Adaboost model with all its default parameters, what's the score ?


In [75]:


ada_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", AdaBoostRegressor())  # default parameters
])
ada_model.fit(X_train, y_train)
from sklearn.metrics import r2_score

y_pred_train = ada_model.predict(X_train)
y_pred_test = ada_model.predict(X_test)

print("R2 AdaBoost train:", r2_score(y_train, y_pred_train))
print("R2 AdaBoost test :", r2_score(y_test, y_pred_test))



R2 AdaBoost train: 0.6446202373235315
R2 AdaBoost test : 0.6607104264758034


15. Train an XGBoost model with all its default parameters except max_depth=3 (the same as adaboost default), what's the score ?

In [76]:

xgb_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", XGBRegressor(
        max_depth=3,       # requested setting
        n_estimators=100,  # XGBoost default
        learning_rate=0.1, # default
        subsample=1,       # default
        colsample_bytree=1,
        objective="reg:squarederror",
        random_state=1
    ))
])
xgb_model.fit(X_train, y_train)
y_pred_train = xgb_model.predict(X_train)
y_pred_test = xgb_model.predict(X_test)

print("R2 XGBoost train:", r2_score(y_train, y_pred_train))
print("R2 XGBoost test :", r2_score(y_test, y_pred_test))



R2 XGBoost train: 0.7867779140743326
R2 XGBoost test : 0.7447112908289639


16. Adaboost does not seem to be performing as well as XGBoost, however it does not seem to overfit the data as much, try and improve it by playing with its parameters ```learning rate``` & ```n_estimators``` thanks to a grid search

In [77]:

ada_pipe = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", AdaBoostRegressor(random_state=1))
])
param_grid = {
    "model__n_estimators": [50, 100, 150, 200],
    "model__learning_rate": [1.0, 0.5, 0.1]
}
grid = GridSearchCV(
    estimator=ada_pipe,
    param_grid=param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)

print("Grid search...")
grid.fit(X_train, y_train)
print("...Done.")
print("Best hyperparameters :", grid.best_params_)
print("Best validation R2 :", grid.best_score_)

best_ada = grid.best_estimator_

y_pred_train = best_ada.predict(X_train)
y_pred_test = best_ada.predict(X_test)

print("R2 on training set :", r2_score(y_train, y_pred_train))
print("R2 on test set :", r2_score(y_test, y_pred_test))



Grid search...
...Done.
Best hyperparameters : {'model__learning_rate': 1.0, 'model__n_estimators': 50}
Best validation R2 : 0.6099214860034713
R2 on training set : 0.6463556605890728
R2 on test set : 0.6529785117287857


We don't seem to be able to reach XGBoost performance using Adaboost in this case

17. Let's now run a sanity check to make sure that Adaboost and XGBoost actually improved the performance of their base models which are regression trees in this case. Train a regression tree model with max_depth = 3 (the default for Adaboost)

In [78]:

tree_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", DecisionTreeRegressor(max_depth=3, random_state=1))
])
tree_model.fit(X_train, y_train)
y_pred_train = tree_model.predict(X_train)
y_pred_test = tree_model.predict(X_test)

print("R2 Tree (depth=3) - train:", r2_score(y_train, y_pred_train))
print("R2 Tree (depth=3) - test :", r2_score(y_test, y_pred_test))


R2 Tree (depth=3) - train: 0.5871131375219447
R2 Tree (depth=3) - test : 0.6153941759705949


We conclude here that both boosting algorithms have fulfilled their missions, they both were able to improve performance on the test set compared to the base model! However XGBoost seems to have superior performance in this case despite higher levels of over fitting.

18. Train separately three independent models, and then implement a voting. Do you get better results?

In [80]:

# méthode de voting :
# combiner 3 modèles complémentaires, non correlés entre eux et de différentes
# familles de modèle que les méthodes d'ensemble
ridge_model = Pipeline(steps=[ # modèle linéaire régularisé
    ("preprocessing", preprocessor),
    ("model", Ridge(alpha=50.0))  # using alpha=50 because grid search usually picks it
])
tree_model = Pipeline(steps=[ # arbre de déicision non linéaire capturant des seuils
    ("preprocessing", preprocessor),
    ("model", DecisionTreeRegressor(max_depth=3, min_samples_split=3, min_samples_leaf=3))
])
svm_model = Pipeline(steps=[ # modèle non linéaire, capturant des frontières complexes différemment des decision tree
    ("preprocessing", preprocessor),
    ("model", SVR(C=1.0, gamma=0.1))
])
ridge_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)
svm_model.fit(X_train, y_train)
print("Ridge R2 test:", ridge_model.score(X_test, y_test))
print("Tree R2 test :", tree_model.score(X_test, y_test))
print("SVM R2 test  :", svm_model.score(X_test, y_test))



Ridge R2 test: 0.7072642228084562
Tree R2 test : 0.6153941759705949
SVM R2 test  : 0.6557510778754636


# Voting

In [84]:

scores_test = {
    "ridge": ridge_model.score(X_test, y_test),
    "tree": tree_model.score(X_test, y_test),
    "svm": svm_model.score(X_test, y_test)
}
best_model = max(scores_test, key=scores_test.get)
best_score = scores_test[best_model]

voting = VotingRegressor(
    estimators=[
        ("ridge", ridge_model),
        ("tree", tree_model),
        ("svm", svm_model)
    ]
)

voting.fit(X_train, y_train)

print("Voting R2 test:", voting.score(X_test, y_test))
print("Voting R2 train:", voting.score(X_train, y_train))

print(f"Best base model in the voting: {best_model} (R2 = {best_score:.4f})")


Voting R2 test: 0.7052977955462789
Voting R2 train: 0.7904132381698371
Best base model in the voting: ridge (R2 = 0.7073)


# Ici le Le meilleur modèle individuel (Ridge) surpasse le Voting

À quoi sert le voting ?

À combiner plusieurs modèles simples pour obtenir une prédiction plus stable, plus robuste, moins sensible aux erreurs individuelles.

Ce qu’il ne fait pas :

Battre systématiquement le meilleur modèle individuel.
Sélectionner le modèle le plus performant.



Parce que Ridge est fortement dominant :

Il capture la majorité du signal

Ses prédictions influencent fortement la moyenne

Le Voting perd un tout petit peu en précision mais gagne en stabilité

Mais comme l’arbre est beaucoup moins bon, il dégrade très légèrement la performance finale.


# Conclusion: voting inutile si modèle dominant comme c'est le cas ici

19. Try a stacking method and conclude about the best model.

1. Le stacking (stacked generalization) sert à :

Combiner plusieurs modèles hétérogènes en construisant un méta-modèle qui apprend comment pondérer intelligemment et corriger les prédictions des modèles de base.

Contrairement au voting :

ce n'est pas une moyenne,

c'est un modèle qui apprend à combiner les autres modèles.

2. Pourquoi le stacking est utile ici ?

Parce que aucun des modèles de base (Ridge, SVM, Tree) ne capture toute la structure des données Airbnb.

Chaque modèle “voit” une partie différente du signal :

In [85]:

# 1️ Définition des modèles de base (identiques à ceux utilisés dans le voting)
ridge_base = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", Ridge(alpha=50.0))
])

tree_base = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", DecisionTreeRegressor(max_depth=3,
                                    min_samples_split=3,
                                    min_samples_leaf=3))
])

svm_base = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", SVR(C=1.0, gamma=0.1))
])

# 2️ Modèle méta (utilisé pour combiner les prédictions des 3 modèles)
meta_model = Ridge(alpha=10.0)

# 3️ Création du stacking
stacking = StackingRegressor(
    estimators=[
        ("ridge", ridge_base),
        ("tree", tree_base),
        ("svm", svm_base)
    ],
    final_estimator=meta_model,
    passthrough=False    # True = ajoute X en input du meta-model (optionnel)
)

# 4️ Entraînement
print("Training stacking regressor...")
stacking.fit(X_train, y_train)
print("...Done.\n")

# 5️ Évaluation
r2_train = stacking.score(X_train, y_train)
r2_test = stacking.score(X_test, y_test)

print("R2 on training set :", r2_train)
print("R2 on test set     :", r2_test)

Training stacking regressor...
...Done.

R2 on training set : 0.8298034330165708
R2 on test set     : 0.7212646797067388


Le stacking surpasse :

tous les modèles individuels

et le voting simple

Il n’atteint pas cependant pas les perf du XGBoost .

**The tree model is very correlated to the other ones. Let's drop it and see if it improves the performances.**

In [86]:
# 1️ Modèles de base restants
ridge_base = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", Ridge(alpha=50.0))
])

svm_base = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", SVR(C=1.0, gamma=0.1))
])

# 2️ Meta-model
meta_model = Ridge(alpha=10.0)

# 3️ Nouveau stacking SANS l'arbre
stacking_no_tree = StackingRegressor(
    estimators=[
        ("ridge", ridge_base),
        ("svm", svm_base)
    ],
    final_estimator=meta_model,
    passthrough=False
)

# 4️ Entraînement
print("Training stacking regressor (no tree)...")
stacking_no_tree.fit(X_train, y_train)
print("...Done.\n")

# 5 Performances
r2_train_nt = stacking_no_tree.score(X_train, y_train)
r2_test_nt  = stacking_no_tree.score(X_test, y_test)

print("Stacking (no tree) - R2 train:", r2_train_nt)
print("Stacking (no tree) - R2 test :", r2_test_nt)


Training stacking regressor (no tree)...
...Done.

Stacking (no tree) - R2 train: 0.8563305120682285
Stacking (no tree) - R2 test : 0.7180671454315937


# XGBoost reste le champion incontesté.

1. En retirant le modèle faible (l’arbre), le stacking repose sur deux modèles plus expressifs :

- Ridge (linéaire mais régularisé)

- SVM (très flexible)

Le méta-modèle trouve alors plus de liberté pour ajuster les prédictions.

➡ Cela augmente la capacité du système → train R² monte
➡ Mais cela n’améliore PAS la généralisation → test R² baisse légèrement

C’est un signe de légère augmentation de variance → début d’overfitting.

\n

 2. Pourquoi le test R² baisse un peu sans l’arbre ?

Même si l’arbre est faible et corrélé aux autres, il apporte :

✔ un signal non linéaire basé sur des interactions simples
✔ un découpage de l’espace de décision très différent de Ridge/SVM
✔ une information complémentaire, même minime

Dans le stacking :

Un modèle faible mais différent améliore souvent l’ensemble.

CQFR : Le stacking ne cherche pas un modèle parfait,
il cherche des modèles diversifiés.

Donc même un arbre peu performant peut légèrement améliorer le test R².